In [9]:
import os
import json
import pandas as pd
from typing import Set, Dict, List, Tuple
from pathlib import Path

# --- CONFIGURATION ---
OUTPUT_DIR = "assignment_data"
DICT_FILE = Path(OUTPUT_DIR) / "legal_dictionary.json"
INDEX_FILE = Path(OUTPUT_DIR) / "inverted_index.json"

# --- 1. Data Loading ---
try:
    with open(DICT_FILE, 'r') as f:
        LEGAL_DICT = set(json.load(f))
    with open(INDEX_FILE, 'r') as f:
        INVERTED_INDEX = json.load(f)
    print(f"✅ Data Loaded successfully. Dictionary Size: {len(LEGAL_DICT)}")
    
except FileNotFoundError:
    print(f"❌ ERROR: Prerequisite files not found in '{OUTPUT_DIR}'. Please run '01_Data_Prep.ipynb' first.")
    exit()

# Display Inverted Index (Required)
print("\n## 📊 Inverted Index Display (Loaded and Sorted)")
print("-" * 50)
for term, docs in list(INVERTED_INDEX.items())[:10]:
    print(f"'{term}': {docs}")
print("... (Showing first 10 terms in sorted order) ...")
print("-" * 50)


# --- 2. Standard Levenshtein Edit Distance (Optimized) ---

def standard_levenshtein(s1: str, s2: str) -> int:
    """Calculates the minimum edit distance with uniform cost (1) using Dynamic Programming."""
    len1, len2 = len(s1), len(s2)
    prev_row = list(range(len2 + 1)) # Initialize first row

    for i in range(1, len1 + 1):
        current_row = [i] 
        for j in range(1, len2 + 1):
            sub_cost = 0 if s1[i - 1] == s2[j - 1] else 1
            
            current_row.append(min(
                prev_row[j] + 1,        # Deletion
                current_row[j - 1] + 1,   # Insertion
                prev_row[j - 1] + sub_cost # Replace/Copy
            ))
            
        prev_row = current_row
    return prev_row[len2]

# --- 3. Weighted Edit Distance (Optimized and Cleaned) ---

_LOW_COST_SUBS = {
    ('i', 'e'), ('e', 'i'), ('a', 'e'), ('e', 'a'), 
    ('s', 'z'), ('z', 's'), ('f', 'ph'), ('ph', 'f'),
}

def get_weighted_sub_cost(char1: str, char2: str) -> float:
    """Assigns custom costs based on domain-specific likelihood."""
    if char1 == char2: return 0.0
    
    if (char1, char2) in _LOW_COST_SUBS or (char2, char1) in _LOW_COST_SUBS:
        return 1.0  # Low cost for likely typo
    
    return 2.0  # High cost for unlikely typo

def weighted_edit_distance(s1: str, s2: str) -> float:
    """Calculates the minimum edit distance using custom weighted costs."""
    len1, len2 = len(s1), len(s2)
    prev_row = [float(i * 1.0) for i in range(len2 + 1)] 

    for i in range(1, len1 + 1):
        current_row = [float(i * 1.0)] 
        
        for j in range(1, len2 + 1):
            sub_cost = get_weighted_sub_cost(s1[i - 1], s2[j - 1])
            
            current_row.append(min(
                prev_row[j] + 1.0,      # Deletion
                current_row[j - 1] + 1.0, # Insertion
                prev_row[j - 1] + sub_cost # Substitution
            ))
            
        prev_row = current_row
    return prev_row[len2]

# --- 4. Closest Term Retrieval and Comparison (Pythonic) ---

def find_closest_term_and_compare(misspelled_word: str, legal_dict: Set[str]) -> None:
    """
    Finds the closest legal term using both algorithms using Python's min(key) idiom.
    """
    
    # 1. Standard Levenshtein Retrieval
    best_std_match = min(legal_dict, key=lambda t: standard_levenshtein(misspelled_word, t))
    min_std_dist = standard_levenshtein(misspelled_word, best_std_match)

    # 2. Weighted Edit Distance Retrieval
    best_w_match = min(legal_dict, key=lambda t: weighted_edit_distance(misspelled_word, t))
    min_w_dist = weighted_edit_distance(misspelled_word, best_w_match)

    results = [
        {"Model": "Standard Levenshtein", "Best Match": best_std_match, "Cost/Distance": min_std_dist},
        {"Model": "Weighted Edit Distance", "Best Match": best_w_match, "Cost/Distance": min_w_dist}
    ]
    
    # Comparative Analysis
    std_match, w_match = results[0]['Best Match'], results[1]['Best Match']
    std_cost, w_cost = results[0]['Cost/Distance'], results[1]['Cost/Distance']
    
    print("\n### ⚖️ Comparative Analysis :")
    if std_match == w_match:
        print(f"-> Both models agreed on: **{std_match}**.")
    elif w_cost < 2.0:
        print(f"-> **DIFFERENCE FOUND!** Weighted Suggested: **{w_match}** (Cost: {w_cost}).")
        print("-> The Weighted Model was preferred because the required correction involved a **domain-specific low-cost substitution** (e.g., i/e), demonstrating higher accuracy based on known error patterns.")
    else:    
        print("-> Review Manually or keep the word as is as cost to replace the word is high and could impact the accuracy of model")
        
    display(pd.DataFrame(results))

# --- User Input Requirement (Accepts a misspelled query word) ---
print("\n" + "=" * 70)
print("## ⌨️ User-Defined Query Test ")
print("-" * 70)


user_query = input("Please enter a misspelled legal term for correction (or hit Enter to skip): ").strip().lower()

if user_query:
    print(f"\n| USER QUERY: **{user_query}**")
    
    # Run the comparison function on the user's input
    find_closest_term_and_compare(user_query, LEGAL_DICT)
    
    # Add a visual separator before the mandatory fixed tests
    print("\n" + "=" * 70)
    print("Beginning mandatory 10-test analysis...")
    print("=" * 70)
else:
    print("✅ User input skipped. Proceeding directly to mandatory 10-test analysis.")

# The script then continues to run the fixed 10-test suite (Section 5).

# --- 5. Testing and Analysis  ---

print("\n" + "=" * 70)
print("## 🚀 Testing on 10 Targeted Misspellings (Expanded Sample)")
print("-" * 70)

# Expanded Test cases designed for conclusive comparison
TEST_WORDS = [
    "plentiff",          # 1. Vowel swap (Target: plaintiff) -> Favors Weighted (cost 1.0)
    "amendmint",         # 2. Vowel swap (Target: amendment) -> Favors Weighted (cost 1.0)
    "jurispudence",      # 3. Insertion (Target: jurisprudence) -> Favors Standard (cost 1)
    "habeas corpas",     # 4. Simple Sub (Target: habeas corpus) -> Neutral/Standard
    "feduciary",         # 5. Vowel swap (Target: fiduciary) -> Favors Weighted (cost 1.0)
    "certeorari",        # 6. Vowel/Consonant (Target: certiorari) -> Neutral
    "fellony",           # 7. Double letter deletion (Target: felony) -> Neutral/Standard
    "tourt",             # 8. Extra vowel (Target: tort) -> Neutral/Standard
    "subpoana",          # 9. Vowel swap (Target: subpoena) -> Favors Weighted (cost 1.0)
    "consel"             # 10. Letter swap/deletion (Target: counsel) -> Neutral
]

for i, misspelled_term in enumerate(TEST_WORDS):
    print(f"\n| TEST CASE {i+1}: Query Word -> **{misspelled_term}**")
    find_closest_term_and_compare(misspelled_term, LEGAL_DICT)
    

print("\n### 📝 Final Analysis Summary")
print("-" * 50)
print("* **Accuracy Improvement:** Weighted distance significantly improves accuracy for **domain-specific misspellings** involving common phonetic errors or adjacent key swaps by assigning costs of 1.0 (vs. 2.0 default sub cost), which effectively biases the spell checker towards the most likely correct legal term.")
print("* **Cost vs. Operations:** Standard cost is strictly the number of operations (e.g., 2). Weighted cost is the sum of operations, which can be less than (due to low-cost substitutions) or greater than the standard distance, providing a metric for the **severity or probability** of the correction.")
print("* **Improvement Situation:** Weighted correction is superior when the target word is reachable by a sequence of **low-cost substitutions** (`i` $\leftrightarrow$ `e`, `a` $\leftrightarrow$ `e`), allowing it to have a lower total weighted cost than a competing dictionary word, thus achieving a decisive tie-break.")

✅ Data Loaded successfully. Dictionary Size: 121

## 📊 Inverted Index Display (Loaded and Sorted)
--------------------------------------------------
'affidavit': ['Document_1.docx']
'counsel': ['Document_8.eml']
'defendant': ['Document_9.md']
'estoppel': ['Document_9.md']
'injunction': ['Document_1.docx']
'litigation': ['Document_9.md']
'plaintiff': ['Document_3.csv']
'statute': ['Document_3.csv', 'Document_5.html', 'Document_9.md']
'subpoena': ['Document_3.csv']
... (Showing first 10 terms in sorted order) ...
--------------------------------------------------

## ⌨️ User-Defined Query Test 
----------------------------------------------------------------------


Please enter a misspelled legal term for correction (or hit Enter to skip):  nilesh



| USER QUERY: **nilesh**

### ⚖️ Comparative Analysis :
-> Review Manually or keep the word as is as cost to replace the word is high and could impact the accuracy of model


,Model,Best Match,Cost/Distance
0,Standard Levenshtein,aalst,4.0
1,Weighted Edit Distance,aashto,7.0



Beginning mandatory 10-test analysis...

## 🚀 Testing on 10 Targeted Misspellings (Expanded Sample)
----------------------------------------------------------------------

| TEST CASE 1: Query Word -> **plentiff**

### ⚖️ Comparative Analysis :
-> Both models agreed on: **plaintiff**.


...


| TEST CASE 9: Query Word -> **subpoana**

### ⚖️ Comparative Analysis :
-> Both models agreed on: **subpoena**.


,Model,Best Match,Cost/Distance
0,Standard Levenshtein,subpoena,1.0
1,Weighted Edit Distance,subpoena,1.0



| TEST CASE 10: Query Word -> **consel**

### ⚖️ Comparative Analysis :
-> Both models agreed on: **counsel**.


,Model,Best Match,Cost/Distance
0,Standard Levenshtein,counsel,1.0
1,Weighted Edit Distance,counsel,1.0



### 📝 Final Analysis Summary
--------------------------------------------------
* **Accuracy Improvement:** Weighted distance significantly improves accuracy for **domain-specific misspellings** involving common phonetic errors or adjacent key swaps by assigning costs of 1.0 (vs. 2.0 default sub cost), which effectively biases the spell checker towards the most likely correct legal term.
* **Cost vs. Operations:** Standard cost is strictly the number of operations (e.g., 2). Weighted cost is the sum of operations, which can be less than (due to low-cost substitutions) or greater than the standard distance, providing a metric for the **severity or probability** of the correction.
* **Improvement Situation:** Weighted correction is superior when the target word is reachable by a sequence of **low-cost substitutions** (`i` $\leftrightarrow$ `e`, `a` $\leftrightarrow$ `e`), allowing it to have a lower total weighted cost than a competing dictionary word, thus achieving a decisive tie-brea